In [1]:
!pip install xgboost lightgbm shap scikit-learn matplotlib seaborn -q

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import IsolationForest, RandomForestClassifier, VotingClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_recall_curve,
    auc,
    balanced_accuracy_score
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import shap

# Fixes the Matplotlib bug natively and applies a clean, publication-grade theme
sns.set_theme(style="whitegrid", context="notebook", palette="muted")
plt.rcParams["figure.figsize"] = (8, 5)
warnings.filterwarnings('ignore')

In [3]:
from google.colab import files

uploaded = files.upload()

Saving EVSE-B-PowerCombined (1).csv to EVSE-B-PowerCombined (1).csv


In [4]:
# Load dataset from Colab environment
df = pd.read_csv("EVSE-B-PowerCombined (1).csv")
print(f"Original Shape: {df.shape}")

# 1. TEMPORAL INTEGRITY CHECK
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values(by='time').reset_index(drop=True)
print(f"Chronological Timeline Cover: {df['time'].min()} to {df['time'].max()}")

# Base physical features
signals = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW"]
window_size = 20

# 2. HIGH-ORDER TRANSIENT ENGINE
print("Engineering high-order statistical and differential transient features...")
for col in signals:
    # Captures localized rate-of-change spikes
    df[f"{col}_delta"] = df[col].diff()

    # Captures localized structural shifts
    df[f"{col}_mean"] = df[col].rolling(window=window_size).mean()
    df[f"{col}_std"] = df[col].rolling(window=window_size).std()

    # Novel Addition: Skewness captures asymmetric wave adjustments caused by line injection attacks
    df[f"{col}_skew"] = df[col].rolling(window=window_size).skew()

    # Crest factor approximation (Peak-to-Average ratio indicating signature distortions)
    df[f"{col}_crest"] = df[col] / (df[f"{col}_mean"] + 1e-5)

# Drop rows with NaN values resulting from looking backward during rolling window operations
df = df.dropna().reset_index(drop=True)
print(f"Shape post-feature engineering: {df.shape}")

Original Shape: (115298, 10)
Chronological Timeline Cover: 2023-12-24 16:18:00 to 2023-12-30 16:20:00
Engineering high-order statistical and differential transient features...
Shape post-feature engineering: (115279, 30)


In [5]:
# Separate out metadata
metadata_cols = ["time", "State", "Attack", "Attack-Group", "Label", "interface"]
feature_cols = [c for c in df.columns if c not in metadata_cols]

X = df[feature_cols]
le_binary = LabelEncoder()
y_binary = le_binary.fit_transform(df["Label"])

le_multi = LabelEncoder()
y_multi = le_multi.fit_transform(df["Attack-Group"])

# --- STRATEGY 1: Pure Chronological (OOD Block Evaluation) ---
split_index = int(len(df) * 0.80)
X_train_chrono, X_test_chrono = X.iloc[:split_index], X.iloc[split_index:]
y_train_bin_c, y_test_bin_c = y_binary[:split_index], y_binary[split_index:]
y_train_mul_c, y_test_mul_c = y_multi[:split_index], y_multi[split_index:]

# --- STRATEGY 2: Stratified Block Split (Cross-Validation Representation) ---
# To prevent leakage, we chunk data into 1-minute blocks, then stratify split the blocks
df['time_block'] = df['time'].dt.to_period('T')
unique_blocks = df['time_block'].unique()

# Assign blocks randomly to train/test to ensure all attack states exist in both sets
from sklearn.model_selection import train_test_split
train_blocks, test_blocks = train_test_split(unique_blocks, test_size=0.20, random_state=42)

train_mask = df['time_block'].isin(train_blocks)
test_mask = df['time_block'].isin(test_blocks)

X_train_strat, X_test_strat = X[train_mask], X[test_mask]
y_train_bin_s, y_test_bin_s = y_binary[train_mask], y_binary[test_mask]
y_train_mul_s, y_test_mul_s = y_multi[train_mask], y_multi[test_mask]

print(f"=== STRATEGY 1 (Chronological) ===\nTest Size: {len(X_test_chrono)} | Unique Test Target Classes: {np.unique(y_test_bin_c)}")
print(f"\n=== STRATEGY 2 (Stratified Session) ===\nTest Size: {len(X_test_strat)} | Unique Test Target Classes: {np.unique(y_test_bin_s)}")

=== STRATEGY 1 (Chronological) ===
Test Size: 23056 | Unique Test Target Classes: [0]

=== STRATEGY 2 (Stratified Session) ===
Test Size: 23013 | Unique Test Target Classes: [0 1]


In [6]:
# Re-instantiate the Hybrid Ensemble Stacking Classifier
def get_fresh_ensemble():
    return VotingClassifier(
        estimators=[
            ('xgb', XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.05, eval_metric="logloss", random_state=42)),
            ('lgb', LGBMClassifier(n_estimators=100, max_depth=5, learning_rate=0.05, random_state=42, verbose=-1)),
            ('rf', RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1))
        ],
        voting='soft'
    )

# Experiment 1: Pure Chronological
ensemble_chrono = get_fresh_ensemble()
ensemble_chrono.fit(X_train_chrono, y_train_bin_c)
preds_c = ensemble_chrono.predict(X_test_chrono)

# Experiment 2: Stratified Session (Leakage-Free representation)
ensemble_strat = get_fresh_ensemble()
ensemble_strat.fit(X_train_strat, y_train_bin_s)
preds_s = ensemble_strat.predict(X_test_strat)

print("\n" + "="*15 + " STRATEGY 1: PURE CHRONOLOGICAL RESULTS " + "="*15)
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test_bin_c, preds_c):.4f}")
print(classification_report(y_test_bin_c, preds_c, target_names=le_binary.classes_, zero_division=0))

print("\n" + "="*15 + " STRATEGY 2: STRATIFIED BLOCK RESULTS " + "="*15)
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test_bin_s, preds_s):.4f}")
print(classification_report(y_test_bin_s, preds_s, target_names=le_binary.classes_, zero_division=0))


=============== STRATEGY 1: PURE CHRONOLOGICAL RESULTS ===============
Balanced Accuracy: 0.7087
              precision    recall  f1-score   support

      attack       1.00      0.71      0.83     23056
      benign       0.00      0.00      0.00         0

    accuracy                           0.71     23056
   macro avg       0.50      0.35      0.41     23056
weighted avg       1.00      0.71      0.83     23056


=============== STRATEGY 2: STRATIFIED BLOCK RESULTS ===============
Balanced Accuracy: 0.9506
              precision    recall  f1-score   support

      attack       0.99      0.99      0.99     19839
      benign       0.96      0.91      0.93      3174

    accuracy                           0.98     23013
   macro avg       0.97      0.95      0.96     23013
weighted avg       0.98      0.98      0.98     23013



In [7]:
xgb_m1 = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.05, objective="multi:softprob", eval_metric="mlogloss", random_state=42)
xgb_m2 = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.05, objective="multi:softprob", eval_metric="mlogloss", random_state=42)

# Fit both strategies
xgb_m1.fit(X_train_chrono, y_train_mul_c)
xgb_m2.fit(X_train_strat, y_train_mul_s)

m_preds_c = xgb_m1.predict(X_test_chrono)
m_preds_s = xgb_m2.predict(X_test_strat)

print("\n" + "="*15 + " MULTICLASS STRATEGY 1 (CHRONOLOGICAL) " + "="*15)
print(classification_report(y_test_mul_c, m_preds_c, target_names=le_multi.classes_, zero_division=0))

print("\n" + "="*15 + " MULTICLASS STRATEGY 2 (STRATIFIED BLOCK) " + "="*15)
print(classification_report(y_test_mul_s, m_preds_s, target_names=le_multi.classes_, zero_division=0))


=============== MULTICLASS STRATEGY 1 (CHRONOLOGICAL) ===============
              precision    recall  f1-score   support

         DoS       0.00      0.00      0.00         0
 host-attack       1.00      0.08      0.15     23056
        none       0.00      0.00      0.00         0
       recon       0.00      0.00      0.00         0

    accuracy                           0.08     23056
   macro avg       0.25      0.02      0.04     23056
weighted avg       1.00      0.08      0.15     23056


=============== MULTICLASS STRATEGY 2 (STRATIFIED BLOCK) ===============
              precision    recall  f1-score   support

         DoS       0.72      0.54      0.61      6085
 host-attack       0.94      0.95      0.95      6720
        none       0.95      0.91      0.93      3174
       recon       0.67      0.83      0.74      7034

    accuracy                           0.80     23013
   macro avg       0.82      0.81      0.81     23013
weighted avg       0.80      0.80      0